# Unmix on a Colab GPU

Runs the Unmix **backend** on Colab's free T4 GPU. The app itself stays on your computer: start it with the address and key this notebook prints, and use it as usual.

**Steps**
1. *Runtime → Change runtime type → T4 GPU*.
2. Using your own fork? Set `REPO` in step 1 to it. Private repository only: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left bar; a *fine-grained* token with read-only *Contents* access to this one repository is enough). Turn on "Notebook access" for it.
3. *Runtime → Run all*. The first run takes about 5 minutes (install + model downloads).
4. Copy the command printed by step 5 into a terminal on your computer, in the Unmix folder.

**Good to know**
* Your audio is uploaded to this Colab machine through an encrypted Cloudflare tunnel. Everything is deleted when the session ends (*Runtime → Disconnect and delete runtime* to end it now).
* The tunnel address is public, so every request needs the access key printed below. Do not share it, or screenshots of it.
* Free Colab sessions end after at most 12 hours, or sooner when idle, and Colab may stop sessions that serve web apps. Just run the notebook again; you get a new address and key.

In [ ]:
#@title 1. Settings
#@markdown The GitHub repository to run: put your own fork here if you have one (`your-name/unmix`).
REPO = "Harsh1tSh/unmix"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Also install voice splitting and clean-up (about 2 more minutes):
SPLIT_VOICES = True  #@param {type:"boolean"}
#@markdown Keep downloaded models in your Google Drive (about 2.5 GB) so later sessions start faster:
CACHE_MODELS_IN_DRIVE = False  #@param {type:"boolean"}

In [ ]:
#@title 2. Check the GPU
import shutil, subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("No GPU in this runtime. Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", smi.stdout.strip())

# Which PyTorch build: CUDA 13 when the driver supports it (Colab's does). It is the
# only build whose torchvision/torchaudio match on Linux; CUDA 12 is the fallback.
import re
m = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
driver_cuda = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
ACCEL = "cuda" if driver_cuda >= (13, 0) else "cuda12"
print(f"Driver supports CUDA {driver_cuda[0]}.{driver_cuda[1]}: using the {ACCEL} build")
if ACCEL != "cuda":
    print("Note: with this driver, voice splitting and clean-up run on the CPU (everything else on the GPU).")
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

In [ ]:
#@title 3. Get the code
import base64, os, subprocess

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # no secret: fine for a public repository

# The token goes in a one-off header, never into the URL or .git/config.
auth = ["-c", "http.extraHeader=Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()] if token else []
ROOT = "/content/unmix"
if os.path.isdir(f"{ROOT}/.git"):
    cmd = ["git", *auth, "-C", ROOT, "pull", "--ff-only", "origin", BRANCH]
else:
    cmd = ["git", *auth, "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", ROOT]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode != 0:
    hint = "" if token else " For a private repository, add the GITHUB_TOKEN secret (see the top of the notebook)."
    raise SystemExit(f"Could not get {REPO}@{BRANCH}: {r.stderr.strip().splitlines()[-1] if r.stderr.strip() else r.returncode}.{hint}")
print(subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
#@title 4. Install (the PyTorch build chosen in step 2)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.6.8"], check=True)
def sync(path, group):
    # --frozen: exactly the versions in uv.lock, nothing newer.
    subprocess.run(["uv", "sync", "--quiet", "--frozen", "--no-default-groups", "--group", group], cwd=path, check=True)
sync(f"{ROOT}/backend", ACCEL)
print("backend installed")
if SPLIT_VOICES:
    # The voice-splitting environment has a GPU build only for CUDA 13 (backend/speakers/pyproject.toml).
    sync(f"{ROOT}/backend/speakers", "cuda" if ACCEL == "cuda" else "cpu")
    print("voice splitting installed")

In [ ]:
#@title 5. Start the backend and the tunnel
import hashlib, json, os, re, secrets, subprocess, time, urllib.request

MODELS = f"{ROOT}/backend/data/models"
if CACHE_MODELS_IN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS = "/content/drive/MyDrive/unmix-models"  # every file is checked against its pinned SHA-256 before use
os.makedirs(MODELS, exist_ok=True)

# Stop what a previous run of this cell started.
for proc_name in ("BACKEND", "TUNNEL"):
    p = globals().get(proc_name)
    if p is not None and p.poll() is None:
        p.terminate(); p.wait(10)

# cloudflared: pinned release, checked against its published SHA-256.
CF = "/content/cloudflared-2026.10.7"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64", CF + ".part")
    if hashlib.sha256(open(CF + ".part", "rb").read()).hexdigest() != "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872":
        os.remove(CF + ".part")
        raise SystemExit("cloudflared download does not match its checksum; not running it.")
    os.chmod(CF + ".part", 0o755); os.rename(CF + ".part", CF)

tunnel_log = open("/content/tunnel.log", "w")
TUNNEL = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
if not URL:
    raise SystemExit("The tunnel did not start; see /content/tunnel.log")

KEY = secrets.token_urlsafe(32)
env = {
    **os.environ,
    "UNMIX_REMOTE_KEY": KEY,                                   # every request must carry it
    "UNMIX_ALLOWED_HOSTS": "localhost,127.0.0.1,[::1]," + URL.removeprefix("https://"),
    "UNMIX_STEM_TRANSPORT": "flac",                           # lossless, half the download
    "UNMIX_MAX_UPLOAD_MB": "95",                              # Cloudflare's limit is 100 MB
    "UNMIX_MODELS_DIR": MODELS,
    "UNMIX_ACCEL": ACCEL,
}
backend_log = open("/content/backend.log", "w")
BACKEND = subprocess.Popen(
    ["uv", "run", "--no-sync", "--no-default-groups", "--group", ACCEL, "uvicorn", "app.server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=f"{ROOT}/backend", env=env, stdout=backend_log, stderr=subprocess.STDOUT,
)

def health():
    req = urllib.request.Request("http://127.0.0.1:8000/api/health", headers={"X-Unmix-Key": KEY})
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.load(r)

print("Loading models (the first time this downloads them: a few minutes)...")
t0 = time.time()
h = None
while time.time() - t0 < 1800:
    if BACKEND.poll() is not None:
        raise SystemExit("The backend stopped; see /content/backend.log:\n" + open("/content/backend.log").read()[-3000:])
    try:
        h = health()
        if h["models_ready"] or h["status"] == "error":
            break
    except OSError:
        pass
    time.sleep(3)
if not h or h["status"] != "ok":
    # Show the cause right here: the server log has the full traceback.
    noise = ("SyntaxWarning", "re.match(", "autocast(", "@autocast", "GET /api/health")
    log_lines = [ln for ln in open("/content/backend.log", errors="replace").read().splitlines() if not any(n in ln for n in noise)]
    print("\n".join(log_lines[-80:]))
    raise SystemExit(f"Models did not load ({(h or {}).get('models_error')}). The server log is printed above; full file: /content/backend.log")

a = h["accelerator"]
print(f"Ready in {time.time() - t0:.0f} s on {a['summary']}")
for n in a["notes"]:
    print("  note:", n)
print()
print("On your computer, in the Unmix folder, run ONE of these:")
print(f"  Windows:      .\\start.ps1 -Remote {URL}")
print(f"  macOS/Linux:  ./start.sh --remote {URL}")
print("and paste this access key when asked:")
print(f"  {KEY}")
print()
print("Keep this tab open. Cell 6 shows what the server is doing.")

In [ ]:
#@title 6. Watch (stop with the ■ button; the server keeps running)
import json, time, subprocess, urllib.request

def gpu_line():
    q = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    return q.stdout.strip()

seen = 0
try:
    while True:
        if BACKEND.poll() is not None:
            print("The backend stopped. Last log lines:")
            print(open("/content/backend.log").read()[-3000:])
            break
        if TUNNEL.poll() is not None:
            print("The tunnel stopped: run cell 5 again for a new address.")
            break
        lines = open("/content/backend.log").read().splitlines()
        for line in lines[seen:]:
            # Job progress, timings and problems; not every HTTP request.
            if any(k in line for k in ("| stage", "| total", "queued |", "ERROR", "WARNING", "clean", "accelerators")):
                print(line)
        seen = len(lines)
        print(f"\r[{time.strftime('%H:%M:%S')}] GPU {gpu_line()}   ", end="")
        time.sleep(10)
except KeyboardInterrupt:
    print("\nStopped watching (the server is still running).")

### Optional: measure the speed

Runs two 60-second jobs (fast and quality) on this GPU and prints how long each stage took, so you can compare with your computer. Separation does the same work whatever the music sounds like, so a synthetic test clip is enough.

In [ ]:
#@title 7. Benchmark (about 3 minutes)
import io, json, time, urllib.request, uuid
import numpy as np, soundfile as sf

sr = 44100
t = np.arange(60 * sr) / sr
rng = np.random.default_rng(0)
mix = np.stack([0.3 * np.sin(2 * np.pi * 80 * t) + 0.2 * np.sin(2 * np.pi * (300 + 50 * np.sin(t)) * t) + 0.05 * rng.standard_normal(t.size)] * 2, axis=1)
buf = io.BytesIO(); sf.write(buf, mix.astype(np.float32), sr, format="WAV", subtype="PCM_16")

def call(method, path, body=None, ctype=None):
    req = urllib.request.Request("http://127.0.0.1:8000" + path, data=body, method=method, headers={"X-Unmix-Key": KEY, **({"Content-Type": ctype} if ctype else {})})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r) if r.status != 204 else None

for mode in ("fast", "quality"):
    b = uuid.uuid4().hex
    parts = [f"--{b}\r\nContent-Disposition: form-data; name=\"{k}\"\r\n\r\n{v}\r\n".encode() for k, v in (("mode", mode), ("lyrics", "true"), ("split_voices", str(SPLIT_VOICES).lower()))]
    parts.append(f"--{b}\r\nContent-Disposition: form-data; name=\"file\"; filename=\"bench.wav\"\r\nContent-Type: audio/wav\r\n\r\n".encode() + buf.getvalue() + f"\r\n--{b}--\r\n".encode())
    job = call("POST", "/api/jobs", b"".join(parts), f"multipart/form-data; boundary={b}")["job_id"]
    while (j := call("GET", f"/api/jobs/{job}"))["status"] not in ("done", "error"):
        time.sleep(2)
    if j["status"] == "error":
        print(mode, "failed:", j.get("error")); continue
    tm = j["timings"]
    print(f"{mode:8s} total {tm['total']:6.1f} s for 60 s of audio ({tm['total'] / 60:.2f}x the clip length)")
    print("         " + ", ".join(f"{k} {v:.1f}s" for k, v in tm.items() if k != "total"))
    call("DELETE", f"/api/jobs/{job}")
print("accelerator:", call("GET", "/api/health")["accelerator"])